In [ ]:
# =====================================================
# HIGH PERFORMERS PERFORMANCE ENGINE
# Technical Script Documentation
# =====================================================
# This notebook implements a multi-model performance
# evaluation framework used to identify High Performers
# across sales roles (EV, EC, SDR).
#
# The engine calculates several performance rationales
# (R1–R7) using statistical normalization, ranking
# methods, and cluster benchmarking.
#
# Author: Fabiana Kuhlmann
# =====================================================


# =====================================================
# 1. LIBRARIES
# =====================================================
# pandas: data manipulation
# numpy: numerical operations
# datetime: timestamp generation for calculation history

import pandas as pd
import numpy as np
from datetime import datetime


# =====================================================
# 2. GLOBAL CONFIGURATION AND BUSINESS RULES
# =====================================================
# This section defines the main parameters used
# throughout the scoring system.

ARQUIVO_EXCEL = r"C:\Users\FabianaKuhlmann\Downloads\Programa High Performers (Phantom Share) (1).xlsx"

# File where cluster averages are historically stored
ARQUIVO_HISTORICO_MEDIAS = "Historico_Medias_Cluster.csv"


# Excel sheet names
ABA_EVS = "EVs"
ABA_ECS = "ECs"
ABA_SDRS = "SDRs"
ABA_METAS = "%Ating_Metas"
ABA_P10 = "Página10"


# Analysis time window
DATA_INICIO = "2025-01-01"
DATA_FIM = "2025-12-31"


# =====================================================
# STANDARDIZED COLUMN NAMES
# =====================================================

COL_PERIODO = "Período (ajustado)"
COL_FRANQUIA = "Franquia (ajustado)"
COL_CLUSTER = "Cluster (ajustado)"

COL_INDICADOR = "Indicador (Ajustado)"
COL_RESULTADO = "Resultado"
COL_COLABORADOR = "Nome"
COL_STATUS = "STATUS ATUAL"

COL_META_COMERCIAL = "Meta Comercial"
COL_NMRR_UNIDADE = "NMRR"


# =====================================================
# VALID EMPLOYEE STATUS
# =====================================================
# Only active employees participate in the evaluation.

STATUS_VALIDOS = ["ATIVO", "ATIVO DUPLICADO"]


# =====================================================
# PERFORMANCE ELIGIBILITY RULES
# =====================================================

# Minimum revenue share required
SHARE_FIXO_PADRAO = 0.25

# Higher share requirement if employee works alone
TETO_SOLO = 0.60

# Minimum score threshold
SCORE_MINIMO = 70

# SDR specific rules (lead share)
SHARE_FIXO_SDR = 0.50
TETO_SOLO_SDR = 0.70

# Minimum unit performance
PCT_META_MINIMO = 0.6

# Minimum score vs cluster average (Top Talent threshold)
META_VS_MEDIA_MINIMA = 100


# =====================================================
# PERFORMANCE INDICATOR WEIGHTS
# =====================================================

# EV (Sales Executive)
PESOS_EVS = {
    "NMRR": 0.4,
    "Demos": 0.2,
    "Conversão": 0.2,
    "Early Churn": 0.2
}

# EC (Channel Executive)
PESOS_ECS = {
    "NMRR de Contador": 0.4,
    "Leads de Contador": 0.2,
    "Contadores indicando": 0.2,
    "Reuniões": 0.2
}

# SDR (Sales Development)
PESOS_SDRS = {
    "Leads Agendados": 0.50,
    "Leads Trabalhados": 0.25,
    "Taxa de Agendamento (%)": 0.25
}


# =====================================================
# INVERSE PERFORMANCE INDICATORS
# =====================================================
# These metrics are better when LOWER.

INDICADORES_INVERSOS = ["Early Churn"]


# =====================================================
# 3. CLUSTER BENCHMARK CALCULATION
# =====================================================
# This function calculates the average performance
# of each indicator inside each cluster.
#
# These averages become the benchmark reference
# used later in R5 scoring.

def get_medias_cluster(df, indicadores, perfil_nome):

    medias = (
        df[df[COL_INDICADOR].isin(indicadores)]
        .groupby([COL_CLUSTER, COL_INDICADOR])[COL_RESULTADO]
        .mean()
        .reset_index()
    )

    pivot_medias = medias.pivot(
        index=COL_CLUSTER,
        columns=COL_INDICADOR,
        values=COL_RESULTADO
    ).reset_index()

    pivot_medias.insert(0, "Perfil", perfil_nome)

    # Timestamp for historical tracking
    pivot_medias["Data_Calculo"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    return pivot_medias


# =====================================================
# 4. CORE PERFORMANCE SCORING ENGINE
# =====================================================
# This function calculates ALL performance rationales.
#
# R1 → Weighted percentile ranking
# R2 → Simple percentile ranking
# R3 → Weighted normalization vs best performer
# R4 → Simple normalization vs best performer
# R5 → Performance vs cluster average
#
# Each rationale represents a different
# analytical perspective.

def calcular_todos_racionais(df, pesos_dict, df_bench):

    indicadores = list(pesos_dict.keys())

    base = df.pivot_table(
        index=[COL_COLABORADOR, COL_FRANQUIA, COL_CLUSTER, COL_PERIODO],
        columns=COL_INDICADOR,
        values=COL_RESULTADO,
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    # Average metric performance per employee
    res = base.groupby(
        [COL_COLABORADOR, COL_FRANQUIA, COL_CLUSTER],
        as_index=False
    ).agg(
        **{
            ind: (ind, "mean")
            for ind in indicadores
            if ind in base.columns
        }
    )


    # =====================================================
    # PERCENTILE AND NORMALIZATION CALCULATIONS
    # =====================================================

    for ind in indicadores:

        if ind in base.columns:

            # Ranking direction
            rank_asc = False if ind in INDICADORES_INVERSOS else True

            # Percentile ranking (R1 and R2)
            base[f"pct_{ind}"] = (
                base.groupby([COL_CLUSTER, COL_PERIODO])[ind]
                .rank(pct=True, ascending=rank_asc)
                * 100
            )

            # Normalization vs best performer (R3 and R4)
            if ind in INDICADORES_INVERSOS:

                base[f"norm_{ind}"] = base.groupby(
                    [COL_CLUSTER, COL_PERIODO]
                )[ind].transform(
                    lambda x: (1 - (x / x.max())) * 100
                    if x.max() != 0 else 100
                )

            else:

                base[f"norm_{ind}"] = base.groupby(
                    [COL_CLUSTER, COL_PERIODO]
                )[ind].transform(
                    lambda x: (x / x.max() * 100)
                    if x.max() != 0 else 0
                )


    # =====================================================
    # PERFORMANCE VS CLUSTER AVERAGE (R5)
    # =====================================================

    res = res.merge(df_bench, on=COL_CLUSTER, suffixes=('_colab', '_cluster'))

    for ind in indicadores:

        if ind in INDICADORES_INVERSOS:

            res[f"{ind}_vs_Media_Cluster"] = np.where(
                res[f"{ind}_colab"] == 0,
                150,
                (res[f"{ind}_cluster"] / res[f"{ind}_colab"]) * 100
            )

        else:

            res[f"{ind}_vs_Media_Cluster"] = (
                res[f"{ind}_colab"] / res[f"{ind}_cluster"]
            ) * 100


    # =====================================================
    # FINAL PERFORMANCE SCORES
    # =====================================================

    res["score_R1_percentil_ponderado"] = sum(
        res[f"pct_{ind}"] * pesos_dict[ind]
        for ind in indicadores
        if f"pct_{ind}" in res.columns
    )

    res["score_R2_percentil_simples"] = res[
        [f"pct_{ind}" for ind in indicadores if f"pct_{ind}" in res.columns]
    ].mean(axis=1)


    res["score_R3_direto_ponderado"] = sum(
        res[f"norm_{ind}"] * pesos_dict[ind]
        for ind in indicadores
        if f"norm_{ind}" in res.columns
    )

    res["score_R4_direto_simples"] = res[
        [f"norm_{ind}" for ind in indicadores if f"norm_{ind}" in res.columns]
    ].mean(axis=1)


    res["score_R5_vs_media_cluster_ponderado"] = sum(
        res[f"{ind}_vs_Media_Cluster"] * pesos_dict[ind]
        for ind in indicadores
        if f"{ind}_vs_Media_Cluster" in res.columns
    )


    res["score_R5_vs_media_cluster_simples"] = res[
        [
            f"{ind}_vs_Media_Cluster"
            for ind in indicadores
            if f"{ind}_vs_Media_Cluster" in res.columns
        ]
    ].mean(axis=1)


    return res


# =====================================================
# 5. DATA LOADING
# =====================================================
# Load all Excel sheets containing operational data.

df_evs = pd.read_excel(ARQUIVO_EXCEL, sheet_name=ABA_EVS)
df_ecs = pd.read_excel(ARQUIVO_EXCEL, sheet_name=ABA_ECS)
df_sdrs = pd.read_excel(ARQUIVO_EXCEL, sheet_name=ABA_SDRS)

df_metas = pd.read_excel(ARQUIVO_EXCEL, sheet_name=ABA_METAS)
df_p10 = pd.read_excel(ARQUIVO_EXCEL, sheet_name=ABA_P10)


# =====================================================
# 6. DATA FILTERING
# =====================================================
# Filters the data according to:
#
# • Time window
# • Valid employee status

for df in [df_evs, df_ecs, df_sdrs]:

    df[COL_PERIODO] = pd.to_datetime(df[COL_PERIODO], errors="coerce")

    df = df[
        df[COL_PERIODO].between(DATA_INICIO, DATA_FIM)
        & df[COL_STATUS].isin(STATUS_VALIDOS)
    ]


# =====================================================
# 7. UNIT TARGET EVALUATION
# =====================================================
# Determine if each business unit reached
# the commercial target.

df_metas["bateu_meta_mes"] = (
    df_metas[COL_NMRR_UNIDADE] /
    df_metas[COL_META_COMERCIAL]
) >= 1


meta_periodo = df_metas.groupby(
    COL_FRANQUIA,
    as_index=False
).agg(

    meses_total=(COL_PERIODO, "nunique"),
    meses_com_meta=("bateu_meta_mes", "sum"),
    nmrr_unidade_total=(COL_NMRR_UNIDADE, "sum")

)

meta_periodo["pct_ating_meta_unidade"] = (
    meta_periodo["meses_com_meta"] /
    meta_periodo["meses_total"]
)

meta_periodo["unidade_bateu_meta"] = np.where(
    meta_periodo["pct_ating_meta_unidade"] >= PCT_META_MINIMO,
    "Sim",
    "Não"
)


# =====================================================
# 8. CLUSTER BENCHMARK CALCULATION
# =====================================================

medias_ev = get_medias_cluster(df_evs, list(PESOS_EVS.keys()), "EV")
medias_ec = get_medias_cluster(df_ecs, list(PESOS_ECS.keys()), "EC")
medias_sdr = get_medias_cluster(df_sdrs, list(PESOS_SDRS.keys()), "SDR")

df_medias_cluster = pd.concat(
    [medias_ev, medias_ec, medias_sdr],
    ignore_index=True
)


# Save benchmark history
df_medias_cluster.to_csv(ARQUIVO_HISTORICO_MEDIAS, index=False)


# =====================================================
# 9. PERFORMANCE SCORE CALCULATION
# =====================================================

score_evs = calcular_todos_racionais(df_evs, PESOS_EVS, medias_ev)
score_ecs = calcular_todos_racionais(df_ecs, PESOS_ECS, medias_ec)
score_sdrs = calcular_todos_racionais(df_sdrs, PESOS_SDRS, medias_sdr)


# =====================================================
# 10. FINAL REPORT EXPORT
# =====================================================
# Generates a multi-sheet Excel file containing
# all analytics outputs.

with pd.ExcelWriter(
    "Relatorio_Performance_Multirracial_Final_Organizado2.xlsx",
    engine="xlsxwriter"
) as writer:

    score_evs.to_excel(writer, sheet_name="Scores_EV", index=False)
    score_ecs.to_excel(writer, sheet_name="Scores_EC", index=False)
    score_sdrs.to_excel(writer, sheet_name="Scores_SDR", index=False)

    df_medias_cluster.to_excel(writer, sheet_name="Cluster_Averages", index=False)


print("✅ Performance Report Successfully Generated!")